In [10]:
# Phase 1 (Transfer Learning): Model initialization only
# - Backbone: MobileNetV2 (ImageNet pretrained)
# - Classes: happy / neutral / sad (3 classes)
# - Input: 224x224 RGB
#
# Academic note: This phase intentionally excludes any dataset loading/training.

import tensorflow as tf
from tensorflow.keras import layers


In [11]:
# 1) Load MobileNetV2 backbone (ImageNet pretrained)
#    - include_top=False removes the 1000-class ImageNet classifier head
#    - alpha=1.0 keeps the standard MobileNetV2 width multiplier
base_model = tf.keras.applications.MobileNetV2(
    include_top=False,
    weights="imagenet",
    alpha=1.0,
    input_shape=(224, 224, 3),
)

In [12]:
# 2) Freeze the entire backbone (transfer learning Phase 1 requirement)
base_model.trainable = False

In [13]:
# 3) Build the new classification head for 3 emotion classes
#    Head: GlobalAveragePooling2D -> Dropout(0.3) -> Dense(3, softmax)
inputs = tf.keras.Input(shape=(224, 224, 3), name="image")

# MobileNetV2 expects inputs preprocessed this way
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)

# Forward pass through frozen backbone
x = base_model(x, training=False)

# Custom head
x = layers.GlobalAveragePooling2D(name="gap")(x)
x = layers.Dropout(0.3, name="dropout_0_3")(x)
outputs = layers.Dense(3, activation="softmax", name="emotion_logits")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name="emotion_mobilenetv2_phase1")

In [14]:
# 4) Compile (Phase 1 requirement)
# - Optimizer: Adam
# - Learning rate: 0.001
# - Loss: sparse_categorical_crossentropy (expects integer class labels: 0/1/2)
# - Metric: accuracy
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

In [15]:
# 5) Inspect architecture (required)
model.summary()

# 6) Save the Phase 1 model (required)
# This saves the full Keras model (architecture + weights).
model.save("emotion_model_phase1.h5")
print("Saved model to emotion_model_phase1.h5")

Model: "emotion_mobilenetv2_phase1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_1 (TrueDivide)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract_1 (Subtract)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gap (GlobalAveragePooling2D)    │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_0_3 (Dropout)           │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ emotion_logits (Dense)          │ (None, 3)              │         3,843 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,261,827 (8.63 MB)

 Trainable params: 3,843 (15.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Saved model to emotion_model_phase1.h5


## Phase 2 (later)

Dataset loading, preprocessing, and fine-tuning will be added in a later phase.

**Per Phase 1 requirements, no dataset code is included here.**